# NLP-to-SQL Prototype (Rule-Based) + PostgreSQL

Prototype ระบบแปลงคำถามภาษาธรรมชาติ (ไทย/อังกฤษแบบง่าย) ให้เป็นคำสั่ง SQL โดยใช้กฎ (rule-based:
regex + keyword matching) ไม่ใช้ ML/NLP library ใดๆ แล้วรันกับฐานข้อมูล PostgreSQL จริง

**ขอบเขต:** รองรับระบบคลังสินค้า (warehouse) 5 ตาราง (suppliers, warehouses, products, inventory,
stock_movements) และคำถามแบบง่าย เช่น
- "มีสินค้ากี่รายการ"
- "สินค้าคงเหลือทั้งหมดกี่ชิ้น"
- "แสดงสินค้าของผู้จำหน่ายชื่อไทยพาร์ทส์"

Schema จะถูกดึงจากฐานข้อมูลจริงทุกครั้งที่รัน (ขั้นตอนที่ 2) ดังนั้นถ้าแก้ตาราง/คอลัมน์ในฐานข้อมูล
prototype จะปรับ SQL ที่ generate ให้ตรงกับโครงสร้างจริงโดยอัตโนมัติ (ไม่ต้องแก้โค้ดส่วน schema เอง)

**ก่อนรัน:** ติดตั้ง dependency ที่จำเป็นก่อน (ครั้งเดียว):
```
pip install psycopg2-binary pandas sqlalchemy
```


## 1. Setup & Connection

Import library ที่จำเป็น และตั้งค่าการเชื่อมต่อ PostgreSQL (แก้ไขค่าใน `DB_CONFIG` ให้ตรงกับฐานข้อมูลของคุณ)

In [30]:
import psycopg2
import psycopg2.extras
import pandas as pd
import re
import json
from datetime import datetime
from sqlalchemy import create_engine

# ----- แก้ไข connection parameters ตรงนี้ให้ตรงกับฐานข้อมูลของคุณ -----
DB_CONFIG = {
    'host': 'localhost',
    'port': 5433,
    'dbname': 'seminarSQL',
    'user': 'postgres',
    'password': 'postgres',
}


def get_connection():
    '''สร้างการเชื่อมต่อไปยัง PostgreSQL ด้วยพารามิเตอร์ที่กำหนดใน DB_CONFIG (ใช้กับ psycopg2 โดยตรง เช่น ขั้นตอน schema extraction)'''
    return psycopg2.connect(**DB_CONFIG)


def get_engine():
    '''สร้าง SQLAlchemy engine จาก DB_CONFIG เดียวกัน (ใช้กับ pd.read_sql_query เพื่อไม่ให้ pandas เตือน
    ว่า DBAPI2 connection ธรรมดาไม่ใช่ SQLAlchemy connectable)'''
    url = f"postgresql+psycopg2://{DB_CONFIG['user']}:{DB_CONFIG['password']}@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['dbname']}"
    return create_engine(url)


try:
    conn = get_connection()
    engine = get_engine()
    print('เชื่อมต่อฐานข้อมูลสำเร็จ')
except Exception as e:
    conn = None
    engine = None
    print(f'เชื่อมต่อฐานข้อมูลไม่สำเร็จ (จะสาธิตด้วย schema ตัวอย่างแทน): {e}')


เชื่อมต่อฐานข้อมูลสำเร็จ


## 2. Schema Extraction

ดึงโครงสร้างฐานข้อมูลจริง (ชื่อตาราง, คอลัมน์, data type, primary key, foreign key) จาก `information_schema` แล้วเก็บเป็น dictionary/JSON เพื่อใช้เป็น "schema reference" ในการสร้าง SQL ขั้นตอนถัดไป

In [31]:
def extract_schema(connection, schema_name='public'):
    '''
    ดึงโครงสร้างฐานข้อมูล (ตาราง, คอลัมน์, ชนิดข้อมูล, primary key, foreign key, comment/alias)
    จาก information_schema + pg_catalog แล้วคืนค่าเป็น dictionary

    'comment' ของแต่ละตาราง/คอลัมน์ (ถ้ามี) มาจาก COMMENT ON TABLE/COLUMN ที่ตั้งไว้ในฐานข้อมูล
    (ดู mock_warehouse_data.sql) ใช้เป็นแหล่งคำพ้อง (alias) ไทย/อังกฤษ แทนการ hardcode
    TABLE_SYNONYMS/COLUMN_SYNONYMS ในโค้ด Python (ขั้นตอนที่ 3 จะนำมาแปลงเป็น dict ต่อ)
    '''
    schema = {}
    if connection is None:
        return schema

    with connection.cursor(cursor_factory=psycopg2.extras.RealDictCursor) as cur:
        # ดึงรายชื่อตารางและคอลัมน์ทั้งหมดใน schema
        cur.execute('''
            SELECT table_name, column_name, data_type, is_nullable
            FROM information_schema.columns
            WHERE table_schema = %s
            ORDER BY table_name, ordinal_position;
        ''', (schema_name,))
        for row in cur.fetchall():
            t = row['table_name']
            schema.setdefault(t, {'columns': [], 'primary_key': [], 'foreign_keys': [], 'comment': None})
            schema[t]['columns'].append({
                'name': row['column_name'],
                'data_type': row['data_type'],
                'nullable': row['is_nullable'] == 'YES',
                'comment': None,
            })

        # ดึง primary key ของแต่ละตาราง
        cur.execute('''
            SELECT tc.table_name, kcu.column_name
            FROM information_schema.table_constraints tc
            JOIN information_schema.key_column_usage kcu
              ON tc.constraint_name = kcu.constraint_name
             AND tc.table_schema = kcu.table_schema
            WHERE tc.constraint_type = 'PRIMARY KEY'
              AND tc.table_schema = %s;
        ''', (schema_name,))
        for row in cur.fetchall():
            t = row['table_name']
            if t in schema:
                schema[t]['primary_key'].append(row['column_name'])

        # ดึง foreign key: ตาราง/คอลัมน์ต้นทาง -> ตาราง/คอลัมน์ปลายทาง
        cur.execute('''
            SELECT
                tc.table_name AS source_table,
                kcu.column_name AS source_column,
                ccu.table_name AS target_table,
                ccu.column_name AS target_column
            FROM information_schema.table_constraints tc
            JOIN information_schema.key_column_usage kcu
              ON tc.constraint_name = kcu.constraint_name
             AND tc.table_schema = kcu.table_schema
            JOIN information_schema.constraint_column_usage ccu
              ON tc.constraint_name = ccu.constraint_name
             AND tc.table_schema = ccu.table_schema
            WHERE tc.constraint_type = 'FOREIGN KEY'
              AND tc.table_schema = %s;
        ''', (schema_name,))
        for row in cur.fetchall():
            t = row['source_table']
            if t in schema:
                schema[t]['foreign_keys'].append({
                    'column': row['source_column'],
                    'ref_table': row['target_table'],
                    'ref_column': row['target_column'],
                })

        # ดึง comment ของตาราง (COMMENT ON TABLE ...) ผ่าน pg_catalog เนื่องจาก
        # information_schema ไม่มี comment ให้ใช้โดยตรง
        cur.execute('''
            SELECT c.relname AS table_name, obj_description(c.oid) AS table_comment
            FROM pg_class c
            JOIN pg_namespace n ON n.oid = c.relnamespace
            WHERE n.nspname = %s AND c.relkind = 'r';
        ''', (schema_name,))
        for row in cur.fetchall():
            t = row['table_name']
            if t in schema:
                schema[t]['comment'] = row['table_comment']

        # ดึง comment ของคอลัมน์ (COMMENT ON COLUMN ...) เช่นเดียวกัน
        cur.execute('''
            SELECT c.relname AS table_name, a.attname AS column_name,
                   col_description(c.oid, a.attnum) AS column_comment
            FROM pg_class c
            JOIN pg_namespace n ON n.oid = c.relnamespace
            JOIN pg_attribute a ON a.attrelid = c.oid
            WHERE n.nspname = %s AND c.relkind = 'r'
              AND a.attnum > 0 AND NOT a.attisdropped;
        ''', (schema_name,))
        for row in cur.fetchall():
            t = row['table_name']
            if t in schema:
                for col in schema[t]['columns']:
                    if col['name'] == row['column_name']:
                        col['comment'] = row['column_comment']
                        break

    return schema


DB_SCHEMA = extract_schema(conn)


In [32]:
# Schema ตัวอย่าง (fallback) ใช้กรณีไม่มีการเชื่อมต่อฐานข้อมูลจริง เพื่อให้ notebook รันสาธิตได้ครบ pipeline
# โครงสร้างสมมติ: ระบบคลังสินค้า (warehouse) 5 ตาราง ตรงกับ mock_warehouse_data.sql
# suppliers -> products -> inventory <- warehouses, products -> stock_movements <- warehouses
# ฟิลด์ 'comment' จำลอง COMMENT ON TABLE/COLUMN ของฐานข้อมูลจริง เพื่อให้ build_table_synonyms/
# build_column_synonyms (ขั้นตอนที่ 3) ทำงานเหมือนกันไม่ว่าจะต่อฐานข้อมูลจริงได้หรือไม่
SAMPLE_SCHEMA = {
    'suppliers': {
        'comment': 'ผู้จำหน่าย|ซัพพลายเออร์|supplier',
        'columns': [
            {'name': 'id', 'data_type': 'integer', 'nullable': False, 'comment': None},
            {'name': 'name', 'data_type': 'character varying', 'nullable': False, 'comment': None},
            {'name': 'contact_name', 'data_type': 'character varying', 'nullable': True, 'comment': None},
            {'name': 'phone', 'data_type': 'character varying', 'nullable': True, 'comment': None},
            {'name': 'email', 'data_type': 'character varying', 'nullable': True, 'comment': None},
        ],
        'primary_key': ['id'],
        'foreign_keys': [],
    },
    'warehouses': {
        'comment': 'คลังสินค้า|คลัง|warehouse',
        'columns': [
            {'name': 'id', 'data_type': 'integer', 'nullable': False, 'comment': None},
            {'name': 'name', 'data_type': 'character varying', 'nullable': False, 'comment': None},
            {'name': 'location', 'data_type': 'character varying', 'nullable': True, 'comment': None},
            {'name': 'capacity', 'data_type': 'integer', 'nullable': True, 'comment': 'ความจุ'},
        ],
        'primary_key': ['id'],
        'foreign_keys': [],
    },
    'products': {
        'comment': 'สินค้า|product',
        'columns': [
            {'name': 'id', 'data_type': 'integer', 'nullable': False, 'comment': None},
            {'name': 'sku', 'data_type': 'character varying', 'nullable': False, 'comment': None},
            {'name': 'name', 'data_type': 'character varying', 'nullable': False, 'comment': None},
            {'name': 'category', 'data_type': 'character varying', 'nullable': True, 'comment': None},
            {'name': 'unit_price', 'data_type': 'numeric', 'nullable': False, 'comment': 'ราคาต่อหน่วย|ราคา|unit price'},
            {'name': 'supplier_id', 'data_type': 'integer', 'nullable': True, 'comment': None},
        ],
        'primary_key': ['id'],
        'foreign_keys': [
            {'column': 'supplier_id', 'ref_table': 'suppliers', 'ref_column': 'id'},
        ],
    },
    'inventory': {
        'comment': 'สต๊อก|สต็อก|คงเหลือ|stock',
        'columns': [
            {'name': 'id', 'data_type': 'integer', 'nullable': False, 'comment': None},
            {'name': 'product_id', 'data_type': 'integer', 'nullable': True, 'comment': None},
            {'name': 'warehouse_id', 'data_type': 'integer', 'nullable': True, 'comment': None},
            {'name': 'quantity', 'data_type': 'integer', 'nullable': False, 'comment': 'จำนวนคงเหลือ|จำนวน'},
            {'name': 'reorder_level', 'data_type': 'integer', 'nullable': True, 'comment': None},
            {'name': 'last_updated', 'data_type': 'timestamp without time zone', 'nullable': True, 'comment': None},
        ],
        'primary_key': ['id'],
        'foreign_keys': [
            {'column': 'product_id', 'ref_table': 'products', 'ref_column': 'id'},
            {'column': 'warehouse_id', 'ref_table': 'warehouses', 'ref_column': 'id'},
        ],
    },
    'stock_movements': {
        'comment': 'ประวัติการเบิกจ่าย|ประวัติการเคลื่อนไหว|เบิกจ่าย|เคลื่อนไหวสินค้า|movement',
        'columns': [
            {'name': 'id', 'data_type': 'integer', 'nullable': False, 'comment': None},
            {'name': 'product_id', 'data_type': 'integer', 'nullable': True, 'comment': None},
            {'name': 'warehouse_id', 'data_type': 'integer', 'nullable': True, 'comment': None},
            {'name': 'movement_type', 'data_type': 'character varying', 'nullable': False, 'comment': None},
            {'name': 'quantity', 'data_type': 'integer', 'nullable': False, 'comment': 'จำนวนคงเหลือ|จำนวน'},
            {'name': 'reference_note', 'data_type': 'character varying', 'nullable': True, 'comment': None},
            {'name': 'movement_date', 'data_type': 'timestamp without time zone', 'nullable': False, 'comment': None},
        ],
        'primary_key': ['id'],
        'foreign_keys': [
            {'column': 'product_id', 'ref_table': 'products', 'ref_column': 'id'},
            {'column': 'warehouse_id', 'ref_table': 'warehouses', 'ref_column': 'id'},
        ],
    },
}

if not DB_SCHEMA:
    print('ไม่พบ schema จากฐานข้อมูลจริง (ยังไม่ได้เชื่อมต่อ) จะใช้ SAMPLE_SCHEMA แทนเพื่อสาธิตการทำงานของ pipeline')
    DB_SCHEMA = SAMPLE_SCHEMA

print(json.dumps(DB_SCHEMA, indent=2, ensure_ascii=False))


{
  "inventory": {
    "columns": [
      {
        "name": "id",
        "data_type": "integer",
        "nullable": false,
        "comment": null
      },
      {
        "name": "product_id",
        "data_type": "integer",
        "nullable": true,
        "comment": null
      },
      {
        "name": "warehouse_id",
        "data_type": "integer",
        "nullable": true,
        "comment": null
      },
      {
        "name": "quantity",
        "data_type": "integer",
        "nullable": false,
        "comment": "จำนวนคงเหลือ|จำนวน"
      },
      {
        "name": "reorder_level",
        "data_type": "integer",
        "nullable": true,
        "comment": null
      },
      {
        "name": "last_updated",
        "data_type": "timestamp without time zone",
        "nullable": true,
        "comment": null
      }
    ],
    "primary_key": [
      "id"
    ],
    "foreign_keys": [
      {
        "column": "product_id",
        "ref_table": "products",
        "ref_co

## 3. Synonym / Keyword Mapping

`TABLE_SYNONYMS` และ `COLUMN_SYNONYMS` ไม่ได้ hardcode ไว้ในโค้ดแล้ว แต่ **สร้างขึ้นอัตโนมัติจาก
`COMMENT ON TABLE/COLUMN` ที่ตั้งไว้ในฐานข้อมูลจริง** (ดู `mock_warehouse_data.sql`) ผ่าน
`build_table_synonyms` / `build_column_synonyms` ด้านล่าง

**เพิ่ม/แก้ไขคำพ้องได้โดยรัน `COMMENT ON TABLE ... IS 'คำที่1|คำที่2|...'` ที่ฐานข้อมูลโดยตรง**
(คั่นแต่ละคำด้วย `|`) ไม่ต้องแก้โค้ด Python หรือ redeploy notebook เลย — เพิ่มตาราง/คอลัมน์ใหม่ก็แค่ตั้ง
comment ให้ตารางนั้น ระบบจะรู้จักคำพ้องทันทีในการรันครั้งถัดไป (ขั้นตอนที่ 2 ดึง schema ใหม่ทุกครั้ง)

ส่วน `INTENT_KEYWORDS` (COUNT/SUM/SELECT) ยังคง hardcode ไว้ในโค้ด เพราะเป็นคำที่บ่งบอกการกระทำ
ทั่วไป ไม่ผูกกับตาราง/คอลัมน์ใดตารางหนึ่งโดยเฉพาะ จึงไม่มี "ที่เก็บ" ตามธรรมชาติใน schema

In [33]:
def build_table_synonyms(schema):
    '''
    สร้าง dict คำพ้อง (ไทย/อังกฤษ) -> ชื่อตารางจริง จาก schema['comment'] ของแต่ละตาราง
    (มาจาก COMMENT ON TABLE ในฐานข้อมูล คั่นแต่ละคำด้วย '|') โดยเติมชื่อตารางจริงเป็นคำพ้องให้อัตโนมัติเสมอ
    '''
    synonyms = {}
    for table_name, info in schema.items():
        aliases = [table_name]
        comment = info.get('comment')
        if comment:
            aliases += [a.strip() for a in comment.split('|') if a.strip()]
        for alias in aliases:
            synonyms[alias.lower()] = table_name
    return synonyms


def build_column_synonyms(schema):
    '''
    สร้าง dict คำพ้อง (ไทย/อังกฤษ) -> ชื่อคอลัมน์จริง จาก column['comment'] ของทุกตาราง
    (มาจาก COMMENT ON COLUMN ในฐานข้อมูล คั่นแต่ละคำด้วย '|') ใช้กับ intent แบบ SUM เป็นหลัก
    ถ้าหลายตารางมีคอลัมน์ชื่อเดียวกัน (เช่น quantity) และมี comment ต่างกัน จะรวมคำพ้องจากทุกที่เข้าด้วยกัน
    '''
    synonyms = {}
    for info in schema.values():
        for col in info['columns']:
            aliases = [col['name']]
            comment = col.get('comment')
            if comment:
                aliases += [a.strip() for a in comment.split('|') if a.strip()]
            for alias in aliases:
                synonyms[alias.lower()] = col['name']
    return synonyms


# ----- แมปคำ (ไทย/อังกฤษ) -> ชื่อตาราง/คอลัมน์จริงในฐานข้อมูล สร้างจาก COMMENT ของ schema จริง -----
TABLE_SYNONYMS = build_table_synonyms(DB_SCHEMA)
COLUMN_SYNONYMS = build_column_synonyms(DB_SCHEMA)

# ----- คำสำคัญที่บ่งบอก intent ของคำถาม (ไม่ผูกกับตาราง/คอลัมน์ใดตารางหนึ่ง จึงยัง hardcode ไว้) -----
INTENT_KEYWORDS = {
    'COUNT': ['กี่คน', 'กี่ชิ้น', 'กี่รายการ', 'กี่แห่ง', 'จำนวนทั้งหมด', 'count', 'how many'],
    'SUM': ['รวมยอด', 'ยอดรวม', 'ทั้งหมดกี่', 'เท่าไหร่', 'sum', 'total'],
    'SELECT': ['แสดง', 'ดู', 'รายการ', 'show', 'list', 'display'],
}

print('TABLE_SYNONYMS:', json.dumps(TABLE_SYNONYMS, indent=2, ensure_ascii=False))
print('COLUMN_SYNONYMS:', json.dumps(COLUMN_SYNONYMS, indent=2, ensure_ascii=False))


TABLE_SYNONYMS: {
  "inventory": "inventory",
  "สต๊อก": "inventory",
  "สต็อก": "inventory",
  "คงเหลือ": "inventory",
  "stock": "inventory",
  "products": "products",
  "สินค้า": "products",
  "product": "products",
  "stock_movements": "stock_movements",
  "ประวัติการเบิกจ่าย": "stock_movements",
  "ประวัติการเคลื่อนไหว": "stock_movements",
  "เบิกจ่าย": "stock_movements",
  "เคลื่อนไหวสินค้า": "stock_movements",
  "movement": "stock_movements",
  "suppliers": "suppliers",
  "ผู้จำหน่าย": "suppliers",
  "ซัพพลายเออร์": "suppliers",
  "supplier": "suppliers",
  "warehouses": "warehouses",
  "คลังสินค้า": "warehouses",
  "คลัง": "warehouses",
  "warehouse": "warehouses"
}
COLUMN_SYNONYMS: {
  "id": "id",
  "product_id": "product_id",
  "warehouse_id": "warehouse_id",
  "quantity": "quantity",
  "จำนวนคงเหลือ": "quantity",
  "จำนวน": "quantity",
  "reorder_level": "reorder_level",
  "last_updated": "last_updated",
  "sku": "sku",
  "name": "name",
  "category": "category",
  "unit_pri

## 4. Rule-Based Intent & Entity Extraction

ใช้ regex/keyword matching (ไม่มี ML) เพื่อจับ (a) intent เช่น COUNT/SUM/SELECT, (b) ตาราง/entity ที่เกี่ยวข้องจาก synonym mapping และ (c) เงื่อนไข filter เช่น ชื่อลูกค้า หรือช่วงวันที่

In [34]:
def detect_intent(text):
    '''ตรวจจับ intent (COUNT, SUM, SELECT) จากคำสำคัญในประโยคคำถาม'''
    text_lower = text.lower()
    for intent, keywords in INTENT_KEYWORDS.items():
        for kw in keywords:
            if kw in text_lower:
                return intent
    return 'SELECT'  # ค่าเริ่มต้นถ้าไม่พบ intent ที่ชัดเจน


def detect_table(text):
    '''ตรวจจับตาราง/entity เป้าหมายของคำถามจาก TABLE_SYNONYMS โดยเลือกคำที่ปรากฏก่อนสุดในประโยค
    (กันปัญหาประโยคที่มีคำพ้องหลายคำ เช่น "สินค้าของผู้จำหน่าย" ต้องเลือก products ไม่ใช่ suppliers)'''
    text_lower = text.lower()
    best_pos, best_table = None, None
    for keyword, table in TABLE_SYNONYMS.items():
        pos = text_lower.find(keyword)
        if pos != -1 and (best_pos is None or pos < best_pos):
            best_pos, best_table = pos, table
    return best_table


def detect_aggregate_column(text):
    '''ตรวจจับคอลัมน์ที่ต้องการคำนวณ (ใช้กับ intent แบบ SUM) จาก COLUMN_SYNONYMS'''
    text_lower = text.lower()
    for keyword, column in COLUMN_SYNONYMS.items():
        if keyword in text_lower:
            return column
    return None


def detect_name_filter(text):
    '''ตรวจจับเงื่อนไขกรองด้วยชื่อที่ระบุในประโยค เช่น "ชื่อสมชาย", "ผู้จำหน่ายชื่อไทยพาร์ทส์" หรือ "named John"
    คืนค่าเป็น dict {'value': ค่าที่ค้นหา, 'entity_table': ตารางที่คำว่า "ชื่อ" อ้างถึง (ถ้าระบุชัดเจน)}
    หรือ None ถ้าไม่พบ

    entity_table ใช้คำพ้อง (TABLE_SYNONYMS) ที่อยู่ *ก่อนหน้า* คำว่า "ชื่อ" ทันที เพื่อรู้ว่าต้องกรองชื่อ
    ของตารางไหน (เช่น "ผู้จำหน่ายชื่อไทยพาร์ทส์" -> suppliers) ถ้าไม่พบคำระบุ entity ชัดเจน จะคืนค่า
    entity_table เป็น None แล้วปล่อยให้ generate_sql ใช้ตารางหลักของคำถามแทน
    '''
    patterns = [
        r'ชื่อ\s*([ก-๙a-zA-Z0-9]+)',
        r'named\s+([a-zA-Z]+)',
        r'name\s+([a-zA-Z]+)',
    ]
    value, match_start = None, None
    for p in patterns:
        m = re.search(p, text, re.IGNORECASE)
        if m:
            value, match_start = m.group(1), m.start()
            break
    if value is None:
        return None

    text_lower = text.lower()
    entity_table, best_pos = None, -1
    for keyword, table in TABLE_SYNONYMS.items():
        pos = text_lower.rfind(keyword, 0, match_start)
        if pos != -1 and pos > best_pos:
            best_pos, entity_table = pos, table

    return {'value': value, 'entity_table': entity_table}


def detect_date_filter(text):
    '''ตรวจจับเงื่อนไขวันที่แบบง่าย เช่น "เดือนนี้" หรือ "this month"'''
    text_lower = text.lower()
    today = datetime.now()
    if 'เดือนนี้' in text_lower or 'this month' in text_lower:
        start = today.replace(day=1).strftime('%Y-%m-%d')
        return {'type': 'month', 'start': start}
    return None


def extract_intent_entities(text):
    '''
    ฟังก์ชันหลักของขั้นตอนที่ 4: รวมผลจากฟังก์ชันตรวจจับด้านบนทั้งหมด
    คืนค่าเป็น dict ที่มี intent, table, aggregate_column, filters
    เพื่อนำไปสร้าง SQL ในขั้นตอนที่ 5 ต่อไป
    '''
    intent = detect_intent(text)
    table = detect_table(text)
    agg_col = detect_aggregate_column(text) if intent == 'SUM' else None
    name_filter = detect_name_filter(text)
    date_filter = detect_date_filter(text)

    filters = {}
    if name_filter:
        filters['name_filter'] = name_filter
    if date_filter:
        filters['date'] = date_filter

    return {
        'raw_text': text,
        'intent': intent,
        'table': table,
        'aggregate_column': agg_col,
        'filters': filters,
    }


# ตัวอย่างการใช้งาน: ทดสอบแปลงคำถามเป็น intent/entity
example = extract_intent_entities('แสดงสินค้าของผู้จำหน่ายชื่อไทยพาร์ทส์')
print(json.dumps(example, indent=2, ensure_ascii=False))


{
  "raw_text": "แสดงสินค้าของผู้จำหน่ายชื่อไทยพาร์ทส์",
  "intent": "SELECT",
  "table": "products",
  "aggregate_column": null,
  "filters": {
    "name_filter": {
      "value": "ไทยพาร์ทส์",
      "entity_table": "suppliers"
    }
  }
}


## 5. SQL Query Generation

แปลง intent + entity ที่ได้จากขั้นตอนที่ 4 ให้เป็น SQL statement จริง โดยอ้างอิงชื่อตาราง/คอลัมน์จาก `DB_SCHEMA` (ขั้นตอนที่ 2) และ join ตารางอัตโนมัติถ้าจำเป็น โดยใช้ `%s` placeholder สำหรับค่าที่มาจากผู้ใช้ เพื่อป้องกัน SQL injection

In [35]:
def find_join_path(schema, table_a, table_b):
    '''หา foreign key ที่เชื่อม table_a กับ table_b โดยตรง คืนค่าเป็น join condition (string) หรือ None ถ้าไม่พบ'''
    if table_a not in schema or table_b not in schema:
        return None
    for fk in schema[table_a].get('foreign_keys', []):
        if fk['ref_table'] == table_b:
            return f"{table_a}.{fk['column']} = {table_b}.{fk['ref_column']}"
    for fk in schema[table_b].get('foreign_keys', []):
        if fk['ref_table'] == table_a:
            return f"{table_b}.{fk['column']} = {table_a}.{fk['ref_column']}"
    return None


# ชื่อคอลัมน์วันที่/เวลาที่พบได้ในตารางต่างๆ ของฐานข้อมูล เรียงเป็นลำดับการค้นหา
DATE_COLUMN_CANDIDATES = ['order_date', 'movement_date', 'last_updated', 'created_at']


def generate_sql(parsed, schema):
    '''
    ขั้นตอนที่ 5: แปลงผลลัพธ์จาก extract_intent_entities ให้เป็น SQL statement จริง
    โดยอ้างอิงชื่อตาราง/คอลัมน์จาก schema ที่ดึงมาได้จริง (ขั้นตอนที่ 2) และ join ตาราง
    อัตโนมัติถ้าต้อง filter ด้วยคอลัมน์ที่อยู่คนละตารางกับตารางหลัก
    คืนค่าเป็น (sql, params, error) โดยใช้ %s placeholder เพื่อป้องกัน SQL injection
    '''
    table = parsed['table']
    intent = parsed['intent']
    filters = parsed['filters']
    agg_col_requested = parsed['aggregate_column']
    params = []

    # ถ้าไม่พบตารางตรงๆ แต่เป็น intent SUM ที่ระบุคอลัมน์ไว้ (เช่น "ราคาเฉลี่ยเท่าไหร่"
    # ไม่มีคำว่าตารางเลย) ให้เดาตารางจากคอลัมน์นั้นโดยค้นหาใน schema ว่าตารางไหนมีคอลัมน์นี้
    if (table is None or table not in schema) and intent == 'SUM' and agg_col_requested:
        for t, info in schema.items():
            if agg_col_requested in [c['name'] for c in info['columns']]:
                table = t
                break

    if table is None or table not in schema:
        return None, params, 'ไม่พบตารางที่เกี่ยวข้องกับคำถามนี้ในฐานข้อมูล'

    joins = []
    where_clauses = []

    # ถ้ามีการระบุชื่อ (เช่น "ผู้จำหน่ายชื่อไทยพาร์ทส์") ให้หาว่าต้องกรองด้วยคอลัมน์ name ของตารางไหน
    # แล้ว join ไปตารางนั้นอัตโนมัติถ้าจำเป็น (ตารางเป้าหมายต้องมีคอลัมน์ name จริงในฐานข้อมูล)
    if 'name_filter' in filters:
        name_val = filters['name_filter']['value']
        entity_table = filters['name_filter']['entity_table']
        # ถ้าไม่ได้ระบุ entity ชัดเจน (เช่นพิมพ์ลอยๆ ว่า "ชื่อสมชาย") ให้ถือว่าหมายถึงตารางหลักของคำถาม
        target_table = entity_table if (entity_table and entity_table in schema) else table

        target_cols = [c['name'] for c in schema[target_table]['columns']]
        if 'name' not in target_cols:
            return None, params, f"ตาราง {target_table} ไม่มีคอลัมน์ 'name' ให้กรองด้วยชื่อได้"

        if target_table == table:
            where_clauses.append(f"{table}.name ILIKE %s")
            params.append(f"%{name_val}%")
        else:
            join_condition = find_join_path(schema, table, target_table)
            if join_condition:
                joins.append(f"JOIN {target_table} ON {join_condition}")
                where_clauses.append(f"{target_table}.name ILIKE %s")
                params.append(f"%{name_val}%")
            else:
                return None, params, f"ไม่พบความสัมพันธ์ (foreign key) ระหว่างตาราง {table} กับ {target_table}"

    # ถ้ามีเงื่อนไขวันที่ และตารางมีคอลัมน์วันที่/เวลาที่รู้จัก ให้เพิ่มเงื่อนไข
    if 'date' in filters:
        date_info = filters['date']
        col_names = [c['name'] for c in schema[table]['columns']]
        date_col = next((c for c in DATE_COLUMN_CANDIDATES if c in col_names), None)
        if date_col:
            where_clauses.append(f"{table}.{date_col} >= %s")
            params.append(date_info['start'])

    # กำหนด SELECT clause ตาม intent ที่ตรวจพบ
    if intent == 'COUNT':
        select_clause = f"SELECT COUNT(*) AS total FROM {table}"
    elif intent == 'SUM':
        col_names = [c['name'] for c in schema[table]['columns']]
        agg_col = agg_col_requested or ('quantity' if 'quantity' in col_names else None)
        if agg_col is None or agg_col not in col_names:
            return None, params, f"ไม่พบคอลัมน์ '{agg_col}' ในตาราง {table}"
        select_clause = f"SELECT SUM({table}.{agg_col}) AS total FROM {table}"
    else:
        select_clause = f"SELECT * FROM {table}"

    sql = select_clause
    if joins:
        sql += ' ' + ' '.join(joins)
    if where_clauses:
        sql += ' WHERE ' + ' AND '.join(where_clauses)
    sql += ';'
    return sql, params, None


# ตัวอย่างการใช้งาน: สร้าง SQL จากคำถามตัวอย่าง แล้ว print ดูก่อน execute จริง
_sql, _params, _err = generate_sql(example, DB_SCHEMA)
print('SQL:', _sql)
print('Params:', _params)
print('Error:', _err)


SQL: SELECT * FROM products JOIN suppliers ON products.supplier_id = suppliers.id WHERE suppliers.name ILIKE %s;
Params: ['%ไทยพาร์ทส์%']
Error: None


## 6. Execute & Display Result

รัน SQL ที่ generate ได้ผ่าน connection แสดงผลเป็น pandas DataFrame และดักจับ error แบบง่ายๆ ไม่ให้ notebook ล่ม

In [36]:
def run_nlp_query(text, connection, schema, sql_engine=None):
    '''
    ขั้นตอนที่ 6: รับคำถามภาษาธรรมชาติ 1 ประโยค แล้วรันผ่านทั้ง pipeline
    (extract_intent_entities -> generate_sql -> execute) และแสดงผลเป็น DataFrame
    มีการดักจับ error แบบง่ายๆ เพื่อไม่ให้ notebook หยุดทำงานกลางคัน

    sql_engine: SQLAlchemy engine (ถ้ามี) ใช้กับ pd.read_sql_query แทน psycopg2 connection ตรงๆ
    เพื่อไม่ให้ pandas เตือนเรื่อง DBAPI2 connection ที่ไม่ใช่ SQLAlchemy connectable
    '''
    parsed = extract_intent_entities(text)
    sql, params, error = generate_sql(parsed, schema)

    print(f"คำถาม: {text}")
    print(f"Intent ที่ตรวจพบ: {parsed['intent']} | ตารางที่ตรวจพบ: {parsed['table']}")

    if error:
        print(f"ไม่สามารถสร้าง SQL ได้: {error}")
        return None

    print(f"SQL ที่ generate:\n{sql}")
    if params:
        print(f"พารามิเตอร์: {params}")

    read_target = sql_engine if sql_engine is not None else connection
    if read_target is None:
        print('ไม่มีการเชื่อมต่อฐานข้อมูลจริง จึงไม่สามารถรัน query ได้ (แสดงเฉพาะ SQL ที่ generate ด้านบน)')
        return None

    try:
        # SQLAlchemy engine ตีความ list ว่าเป็นหลายชุดพารามิเตอร์ (executemany) จึงต้องแปลงเป็น tuple ก่อนเสมอ
        df = pd.read_sql_query(sql, read_target, params=tuple(params))
        print(f"ผลลัพธ์: พบ {len(df)} แถว")
        return df
    except Exception as e:
        print(f"เกิดข้อผิดพลาดขณะรัน query: {e}")
        return None


## 7. Demo Cell

รันคำถามตัวอย่างภาษาไทยหลายข้อผ่าน pipeline ทั้งหมด (ขั้นตอนที่ 4-6) ในเซลล์เดียว เพื่อดู input -> generated SQL -> ผลลัพธ์

In [37]:
demo_questions = [
    'มีสินค้ากี่รายการ',
    'คลังสินค้ามีกี่แห่ง',
    'จำนวนคงเหลือของสินค้าทั้งหมดเท่าไหร่',
    'แสดงสินค้าของผู้จำหน่ายชื่อไทยพาร์ทส์',
    'แสดงคลังสินค้าทั้งหมด',
]

for q in demo_questions:
    print('=' * 70)
    result_df = run_nlp_query(q, conn, DB_SCHEMA, sql_engine=engine)
    if result_df is not None:
        display(result_df)
    print()


คำถาม: มีสินค้ากี่รายการ
Intent ที่ตรวจพบ: COUNT | ตารางที่ตรวจพบ: products
SQL ที่ generate:
SELECT COUNT(*) AS total FROM products;
ผลลัพธ์: พบ 1 แถว


,total
0,15



คำถาม: คลังสินค้ามีกี่แห่ง
Intent ที่ตรวจพบ: COUNT | ตารางที่ตรวจพบ: warehouses
SQL ที่ generate:
SELECT COUNT(*) AS total FROM warehouses;
ผลลัพธ์: พบ 1 แถว


,total
0,4



คำถาม: จำนวนคงเหลือของสินค้าทั้งหมดเท่าไหร่
Intent ที่ตรวจพบ: SUM | ตารางที่ตรวจพบ: inventory
SQL ที่ generate:
SELECT SUM(inventory.quantity) AS total FROM inventory;
ผลลัพธ์: พบ 1 แถว


,total
0,11298



คำถาม: แสดงสินค้าของผู้จำหน่ายชื่อไทยพาร์ทส์
Intent ที่ตรวจพบ: SELECT | ตารางที่ตรวจพบ: products
SQL ที่ generate:
SELECT * FROM products JOIN suppliers ON products.supplier_id = suppliers.id WHERE suppliers.name ILIKE %s;
พารามิเตอร์: ['%ไทยพาร์ทส์%']
ผลลัพธ์: พบ 4 แถว


,id,sku,name,category,unit_price,supplier_id,id,name,contact_name,phone,email
0,1,SKU-0001,สายไฟ THW 2.5 sq.mm.,วัสดุไฟฟ้า,12.5,1,1,บริษัท ไทยพาร์ทส์ จำกัด,สมชาย ใจดี,02-111-2233,contact@thaiparts.co.th
1,2,SKU-0002,เบรกเกอร์ 20A,วัสดุไฟฟ้า,185.0,1,1,บริษัท ไทยพาร์ทส์ จำกัด,สมชาย ใจดี,02-111-2233,contact@thaiparts.co.th
2,14,SKU-0014,สวิตช์ไฟ 2 ทาง,วัสดุไฟฟ้า,35.0,1,1,บริษัท ไทยพาร์ทส์ จำกัด,สมชาย ใจดี,02-111-2233,contact@thaiparts.co.th
3,15,SKU-0015,ปลั๊กพ่วง 4 ช่อง มอก.,วัสดุไฟฟ้า,199.0,1,1,บริษัท ไทยพาร์ทส์ จำกัด,สมชาย ใจดี,02-111-2233,contact@thaiparts.co.th



คำถาม: แสดงคลังสินค้าทั้งหมด
Intent ที่ตรวจพบ: SELECT | ตารางที่ตรวจพบ: warehouses
SQL ที่ generate:
SELECT * FROM warehouses;
ผลลัพธ์: พบ 4 แถว


,id,name,location,capacity
0,1,คลังกรุงเทพ (สำนักงานใหญ่),"บางนา, กรุงเทพมหานคร",50000
1,2,คลังชลบุรี,"ศรีราชา, ชลบุรี",30000
2,3,คลังระยอง,"มาบตาพุด, ระยอง",20000
3,4,คลังขอนแก่น,"เมืองขอนแก่น, ขอนแก่น",15000
